# Sales Forecasting 

## Internship Final Project 

### Imports 

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

import pmdarima
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX

from prophet import Prophet

from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

## Task 1 -  Data Loading, Merging & Deep Exploration


### Loading Dataset

In [ ]:
sales_df = pd.read_csv("train.csv")
games_df = pd.read_csv("vgsales.csv")

In [ ]:
sales_df.head()

### Check shape 

In [ ]:
sales_df.shape

### Check Data types

In [ ]:
sales_df.info()

### Parse Date Columns 

In [ ]:
# Convert Date Columns into datetime format (DD/MM/YYYY)


sales_df["Order Date"] = pd.to_datetime(sales_df["Order Date"],format="%d/%m/%Y")
sales_df["Ship Date"] = pd.to_datetime(sales_df["Ship Date"],format="%d/%m/%Y")

In [ ]:
# Verify the updated data types

sales_df.info()

### Time Feature Extraction 

In [ ]:
# Extract time-based features from the Order Date
sales_df["Year"] = sales_df["Order Date"].dt.year
sales_df["Month"] = sales_df["Order Date"].dt.month
sales_df["Week Number"] = sales_df["Order Date"].dt.isocalendar().week
sales_df["Day of Week"] = sales_df["Order Date"].dt.day_name()
sales_df["Quarter"] = sales_df["Order Date"].dt.quarter

In [ ]:
sales_df['Order Date'].min(),sales_df['Order Date'].max()

### Season Feature

In [ ]:
# Map each month to its corresponding season
season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
}

sales_df["Season"] = sales_df["Month"].map(season_map)

### Data Quality Check

In [ ]:
# check for missing values
sales_df.isnull().sum()

### Observation 

Most of the columns do not contain any missing values. Only the 'Postal Code' column has 11 missing entries, while all other columns are complete.

In [ ]:
# check for duplicate values
sales_df.duplicated().sum()

### Observation

No duplicate records were found in the dataset, indicating that each transaction is unique. This helps ensure that the analysis and forecasting results are not affected by repeated entries.

In [ ]:
sales_df.info()

### Observation

After converting the date columns, all features have appropriate data types. No additional data type issues were found in the dataset apart from the previously identified missing values in the Postal Code column.


In [ ]:
sales_df['Order Date'].duplicated().sum()

### Daily , weekly and Monthly Sales Aggregation

In [ ]:
# Aggregate total sales for each day
daily_sales = (
    sales_df.groupby("Order Date")["Sales"]
    .sum()
    .reset_index()
)

daily_sales.head()

In [ ]:
daily_sales.shape

In [ ]:
# Aggregate daily sales into weekly totals
weekly_sales = (
    daily_sales.set_index("Order Date")
    .resample("W")["Sales"]
    .sum()
    .reset_index()
)

weekly_sales.head()

In [ ]:
# Aggregate daily sales into monthly totals
monthly_sales = (
    daily_sales.set_index("Order Date")
    .resample("ME")["Sales"]
    .sum()
    .reset_index()
)

monthly_sales.head()

In [ ]:
sales_df['Category'].value_counts()

### Revenue Analysis by Product Category

In [ ]:
# Calculate the total revenue generated by each product category
category_revenue = sales_df.groupby("Category")["Sales"].sum().sort_values(ascending=False)

category_revenue.round(2)

### Observation :

The Technology category generated the highest total revenue, contributing approximately $827,455.87 in sales. Furniture ranked second, while Office Supplies generated the lowest total revenue among the three categories.

### Revenue Visualization

In [ ]:
# Visualize total revenue by product category
category_revenue.plot(kind="bar")

plt.title("Total Revenue by Product Category")
plt.xlabel("Category")
plt.ylabel("Total Sales")
plt.xticks(rotation=0)
plt.show()

### Sales Growth By Region

In [ ]:
# Calculate yearly sales for each region
region_sales = sales_df.groupby(["Year", "Region"])["Sales"].sum().unstack()

region_sales

### Sales Growth Visualization

In [ ]:
region_sales.plot(marker="o")

plt.title("Sales Growth by Region")
plt.xlabel("Year")
plt.ylabel("Total Sales")
plt.legend(title="Region")
plt.show()

### Observation : 
The East region showed the most consistent sales growth over the four-year period, with sales increasing steadily each year. Although the West region achieved the highest overall sales by 2018, it experienced a decline between 2015 and 2016, making its growth less consistent than the East region.

### Shipping Time Analysis by Region

In [ ]:
# Calculate the shipping time (in days) for each order
sales_df["Shipping Time"] = (
    sales_df["Ship Date"] - sales_df["Order Date"]
).dt.days

sales_df[["Order Date", "Ship Date", "Shipping Time"]].head()

In [ ]:
# Calculate the average shipping time for each region
shipping_time_by_region = (
    sales_df.groupby("Region")["Shipping Time"]
    .mean()
    .sort_values()
)

shipping_time_by_region.round(2)

### Observation : 
The average shipping time is approximately 4 days across all regions. Although the Central region has the highest average shipping time (4.07 days) and the East region has the lowest (3.91 days), the differences are minimal, indicating a fairly consistent shipping process across regions.

### Monthly Sales Seasonality Analysis

In [ ]:
sales_df.groupby(['Year',"Month"])["Sales"].sum().unstack()

### Visualization

In [ ]:
# Visualize monthly sales trends across different years
monthly_trend = sales_df.groupby(["Year", "Month"])["Sales"].sum().unstack()

monthly_trend.T.plot(figsize=(10, 5), marker="o")

plt.title("Monthly Sales Trend Across Years")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(range(1, 13))
plt.legend(title="Year")
plt.show()

### Observation : 
The sales data shows a clear seasonal pattern across all four years. Sales consistently increase during September, November, and December, with November recording the highest sales in most years. In contrast, the beginning of the year, particularly January and February, generally experiences lower sales. This indicates strong year-end seasonal demand.

##  Task 1 Summary

- Successfully loaded and explored the Superstore Sales dataset.
- Performed feature engineering by extracting time-based features.
- Checked data quality, including missing values, duplicates, and data types.
- Aggregated sales into daily, weekly, and monthly levels.
- Identified Technology as the highest revenue-generating category.
- Found that the East region showed the most consistent sales growth.
- Observed that average shipping time remained close to 4 days across all regions.
- Detected clear seasonality, with sales peaking during September, November, and December.

## Task 2 — Time Series Analysis & Decomposition

### Overall Monthly Sales Trend

In [ ]:
# Visualize the overall monthly sales trend
plt.figure(figsize=(12,5))

plt.plot(monthly_sales['Order Date'] , monthly_sales['Sales'],marker = "o")

plt.title("Overall Monthly Sales Trend (2015 - 2018)")
plt.xlabel("Order Date")
plt.ylabel("Total Monthly Sales")

plt.grid(True)
plt.show()

### Time Series Decomposition

In [ ]:
# Decompose the monthly sales time series into trends , seasonal and residual components
decomposition = seasonal_decompose(
    monthly_sales["Sales"],
    model="additive",
    period=12
)

decomposition.plot()
plt.show()

### Observation : 
1. The trend component shows a gradual upward movement over time, indicating that overall sales have increased during the four-year period.

2. The seasonal component displays a repeating pattern every year, suggesting that the sales data has strong seasonality.

3. The residual component appears to be randomly distributed around zero, indicating that most of the systematic trend and seasonal patterns have been successfully captured by the decomposition process.

4. A few months exhibit relatively high positive and negative residual values, showing occasional unexpected fluctuations in sales that cannot be explained by trend or seasonality alone.

### Stationarity Test (Augmented Dickey-Fuller Test)

In [ ]:
# Perform the ADF test on the monthly sales series
adf_result = adfuller(monthly_sales["Sales"])

print(f"ADF Statistic : {adf_result[0]}")
print(f"P-value       : {adf_result[1]}")
print(f"Used Lags     : {adf_result[2]}")
print(f"Observations  : {adf_result[3]}")

print("\nCritical Values:")
for key, value in adf_result[4].items():
    print(f"{key}: {value}")

### Explanation :
The ADF test is used to check whether a time series is stationary.

If the p-value is less than 0.05, the null hypothesis is rejected, which indicates that the series is stationary.

If the p-value is greater than 0.05, the series is considered non-stationary.

### Observation : 
The p-value obtained from the ADF test is 0.00028, which is less than 0.05.

Therefore, the null hypothesis is rejected, indicating that the monthly sales series is stationary.

Since the series is already stationary, differencing is not required.

In [ ]:
monthly_sales.info()

## Task 3 — Sales Forecasting using 3 Different Models

### Model 1 — SARIMA (Statistical Model)


### Data Preparation for SARIMA

The monthly sales data was prepared for time series forecasting by setting the Order Date column as the index. The resulting time series contains 48 monthly observations from January 2015 to December 2018, making it suitable for SARIMA modeling.

In [ ]:
# Create a copy of the monthly sales data for time series forecasting
ts_data = monthly_sales.copy()

# Set the Order Date as the index
ts_data.set_index("Order Date", inplace=True)


ts_data.head()

### Parameter Selection using Auto ARIMA

In [ ]:
# Automatically determine the best SARIMA parameters
auto_model = auto_arima(
    ts_data["Sales"],
    seasonal=True,
    m=12,                 # Monthly seasonality
    trace=True,
    suppress_warnings=True,
    stepwise=True
)

# Display the selected model
print("Auto Model SUmmary : ",auto_model.summary())

### Explanation : 
The SARIMA parameters were selected using the auto_arima() function from the pmdarima library. This function automatically evaluates multiple combinations of ARIMA and seasonal parameters and selects the model with the lowest Akaike Information Criterion (AIC), which indicates a better balance between model accuracy and complexity. The best model selected was SARIMA(2,1,0)(1,0,0,12) with an AIC score of 1071.915.

### SARIMA Model Traning

In [ ]:
# Train the SARIMA model using the parameters selected by auto_arima
sarima_model = SARIMAX(
    ts_data["Sales"],
    order=(2, 1, 0),
    seasonal_order=(1, 0, 0, 12)
)

sarima_result = sarima_model.fit()

print(sarima_result.summary())

### 3-Month Sales Forecast 

In [ ]:
# Forecast the next 3 months
forecast = sarima_result.get_forecast(steps=3)

# Forecasted values
forecast_mean = forecast.predicted_mean

# Confidence intervals
forecast_ci = forecast.conf_int()

print("Forecasted Sales:")
print(forecast_mean)

print("\nConfidence Intervals:")
print(forecast_ci)

### SARIMA Model Evaluation

In [ ]:
# Predict on the training period
sarima_pred = sarima_result.predict(
    start=ts_data.index[1],
    end=ts_data.index[-1]
)

# Actual values
sarima_actual = ts_data.iloc[1:]

# Calculate metrics
sarima_mae = mean_absolute_error(sarima_actual, sarima_pred)
sarima_rmse = np.sqrt(mean_squared_error(sarima_actual, sarima_pred))
sarima_mape =  np.mean(
    np.abs(
        (sarima_actual["Sales"].values - sarima_pred.values)
        / sarima_actual["Sales"].values
    )
) * 100



print(f"MAE  : {sarima_mae:.2f}")
print(f"RMSE : {sarima_rmse:.2f}")
print(f"MAPE : {sarima_mape:.2f}%")

### Actual Vs Forecasted Sales

In [ ]:
# Plot actual sales and forecasted sales
plt.figure(figsize=(12, 5))

# Historical sales
plt.plot(
    ts_data.index,
    ts_data["Sales"],
    label="Actual Sales",
    marker="o"
)

# Forecast
plt.plot(
    forecast_mean.index,
    forecast_mean,
    label="Forecast",
    marker="o",
    linestyle="--"
)

# Confidence Interval
plt.fill_between(
    forecast_ci.index,
    forecast_ci.iloc[:, 0],
    forecast_ci.iloc[:, 1],
    alpha=0.2,
    label="95% Confidence Interval"
)

plt.title("SARIMA Forecast (Next 3 Months)")
plt.xlabel("Date")
plt.ylabel("Sales")

plt.legend()
plt.grid(True)

plt.show()

### Observation : 
The SARIMA model predicts that sales will remain relatively stable over the next three months, with forecasted values of approximately 71,457, 55,171, and 75,354 respectively. The confidence intervals are fairly wide, indicating a degree of uncertainty in the predictions, which is expected due to the limited size of the historical dataset. Overall, the model captures the general sales trend and seasonal behavior, making it suitable as a baseline statistical forecasting model

## Model 2 — Facebook Prophet (Industry-Standard Forecasting Tool)

### Data Preparation

In [ ]:
prophet_data = monthly_sales.copy()

prophet_data.rename(
    columns={
        "Order Date": "ds",
        "Sales": "y"
    },
    inplace=True
)

# Display
prophet_data.head()

### Training the Prophet Model 

In [ ]:
# Initialize the Prophet model
prophet_model = Prophet()

# Train model
prophet_model.fit(prophet_data)

### Propeht Model Evaluation

In [62]:
# Keeping only the historical predictions
prophet_actual = prophet_data["y"]
prophet_pred = forecast.iloc[:len(prophet_data)]["yhat"]

# Calculate evaluation metrics
prophet_mae = mean_absolute_error(prophet_actual, prophet_pred)
prophet_rmse = np.sqrt(mean_squared_error(prophet_actual, prophet_pred))
prophet_mape = np.mean(
    np.abs((prophet_actual.values - prophet_pred.values) / prophet_actual.values)
) * 100

print(f"MAE  : {prophet_mae:.2f}")
print(f"RMSE : {prophet_rmse:.2f}")
print(f"MAPE : {prophet_mape:.2f}%")

MAE  : 5770.42
RMSE : 7272.00
MAPE : 14.48%


### Generate 3-Month Sales Forecast

In [ ]:
#  dataframe for the next 3 months
future = prophet_model.make_future_dataframe(periods=3, freq="ME")

# Generate predictions
forecast = prophet_model.predict(future)

# Display the last 3 forecasted months
forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(3)

### Prophet Forecast Visualization

In [ ]:
# Plot the forecast
fig1 = prophet_model.plot(forecast)

plt.title("Prophet Sales Forecast")
plt.xlabel("Date")
plt.ylabel("Sales")

plt.show()

### Trend and Seasonality Components

In [ ]:
# Plot trend and seasonality components
fig2 = prophet_model.plot_components(forecast)

plt.show()

### Observation : 
The Prophet model shows an overall upward sales trend over the four-year period, indicating gradual business growth. The yearly seasonality component suggests that sales fluctuate during different months of the year, confirming the presence of seasonal patterns. Since the dataset was aggregated at the monthly level, Prophet did not generate a weekly seasonality component because weekly patterns cannot be learned from monthly observations. The forecast predicts approximately 42,991, 31,248, and 81,267 sales for the next three months.

### Weekly Seasonality Note : 
Weekly seasonality was not generated because the dataset was aggregated to monthly sales before training the Prophet model. Weekly seasonality requires daily or weekly observations to identify recurring weekly patterns.

## Model 3 — XGBoost Regressor 

### Feature Engineerign

In [ ]:
# Create a copy of the monthly sales data
xgb_data = monthly_sales.copy()

# Create lag features
xgb_data["Lag_1"] = xgb_data["Sales"].shift(1)
xgb_data["Lag_2"] = xgb_data["Sales"].shift(2)
xgb_data["Lag_3"] = xgb_data["Sales"].shift(3)

# Create 3-month rolling mean
xgb_data["Rolling_Mean_3"] = xgb_data["Sales"].rolling(window=3).mean()

# Extract calendar features
xgb_data["Month"] = xgb_data["Order Date"].dt.month
xgb_data["Quarter"] = xgb_data["Order Date"].dt.quarter

# Season feature
xgb_data["Season"] = xgb_data["Month"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
})

# Display first rows
xgb_data.head()

### Data Preparation

In [ ]:
# Remove rows with missing values 
xgb_data = xgb_data.dropna().reset_index(drop=True)

# Check the dataset
xgb_data.head()

print("Shape:", xgb_data.shape)

### Encoding Categorical Features

In [ ]:
# Convert the Season column into numerical dummy variables
xgb_data = pd.get_dummies(
    xgb_data,
    columns=["Season"],
    drop_first=True
)

# Display the first few rows
xgb_data.head()

print("Columns:")
print(xgb_data.columns)

### Feature Selection and Train-Test Split


In [ ]:
# Features
X = xgb_data.drop(columns=["Order Date", "Sales"])

# Target
y = xgb_data["Sales"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

### Trainig the XGBOOST Regressor 

In [ ]:
# Initialize the XGBoost model
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Train the model
xgb_model.fit(X_train, y_train)

### Model Prediction

In [ ]:
# Predict sales for the test dataset
y_pred = xgb_model.predict(X_test)

# Display actual vs predicted values
prediction_df = pd.DataFrame({
    "Actual Sales": y_test.values,
    "Predicted Sales": y_pred
})

prediction_df.head()

### Model Evaluation

In [ ]:
# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAPE : {mape:.2f}%")

### Actual Vs Predicted Sales

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    y_test.values,
    label="Actual Sales",
    marker="o"
)

plt.plot(
    y_pred,
    label="Predicted Sales",
    marker="o",
    linestyle="--"
)

plt.title("XGBoost: Actual vs Predicted Sales")
plt.xlabel("Test Observations")
plt.ylabel("Sales")

plt.legend()
plt.grid(True)

plt.show()

### Observation : 
The XGBoost model successfully captures the overall increasing trend in sales but smooths extreme fluctuations. It predicts the general direction of sales reasonably well, although it underestimates the highest sales peak. The model achieved a MAE of 8903.40, RMSE of 13351.83, and MAPE of 12.51%, indicating good forecasting performance on the test dataset.

### Forecasting the Next 3-Months Using XGBoost

In [ ]:
future_predictions = []

# Last known sales values
last_sales = monthly_sales["Sales"].tolist()

# Predict next 3 months
for i in range(3):

    lag_1 = last_sales[-1]
    lag_2 = last_sales[-2]
    lag_3 = last_sales[-3]

    rolling_mean = np.mean([lag_1, lag_2, lag_3])

    month = (12 + i) % 12 + 1
    quarter = (month - 1) // 3 + 1

    season_spring = 1 if month in [3, 4, 5] else 0
    season_summer = 1 if month in [6, 7, 8] else 0
    season_winter = 1 if month in [12, 1, 2] else 0

    future_input = pd.DataFrame({
        "Lag_1": [lag_1],
        "Lag_2": [lag_2],
        "Lag_3": [lag_3],
        "Rolling_Mean_3": [rolling_mean],
        "Month": [month],
        "Quarter": [quarter],
        "Season_Spring": [season_spring],
        "Season_Summer": [season_summer],
        "Season_Winter": [season_winter]
    })

    prediction = xgb_model.predict(future_input)[0]

    future_predictions.append(prediction)
    last_sales.append(prediction)

# Display forecast
forecast_df = pd.DataFrame({
    "Month": ["2019-01", "2019-02", "2019-03"],
    "Forecasted Sales": future_predictions
})

forecast_df

### Model Comparisoon

In [64]:
comparison_df = pd.DataFrame({
    "Model": ["SARIMA", "Prophet", "XGBoost"],
    "MAE": [
        round(sarima_mae, 2),
        round(prophet_mae, 2),
        round(mae, 2)
    ],
    "RMSE": [
        round(sarima_rmse, 2),
        round(prophet_rmse, 2),
        round(rmse, 2)
    ],
    "MAPE": [
        round(sarima_mape, 2),
        round(prophet_mape, 2),
        round(mape, 2)
    ],
    "Forecast Month 1": [
        round(forecast_mean.iloc[0], 2),
        round(forecast["yhat"].iloc[-3], 2),
        round(future_predictions[0], 2)
    ],
    "Forecast Month 2": [
        round(forecast_mean.iloc[1], 2),
        round(forecast["yhat"].iloc[-2], 2),
        round(future_predictions[1], 2)
    ],
    "Forecast Month 3": [
        round(forecast_mean.iloc[2], 2),
        round(forecast["yhat"].iloc[-1], 2),
        round(future_predictions[2], 2)
    ]
})

comparison_df

,Model,MAE,RMSE,MAPE,Forecast Month 1,Forecast Month 2,Forecast Month 3
0,SARIMA,14437.20,19386.18,42.11,71457.290000,55170.680000,75354.270000
1,Prophet,5770.42,7272.00,14.48,42990.530000,31248.160000,81267.010000
2,XGBoost,8903.40,13351.83,12.51,52179.890625,40521.660156,61992.578125


### Best Model Recommendation : 
Based on the evaluation metrics, Prophet is recommended for production use because it achieved the lowest MAE (5770.42) and RMSE (7272.00), indicating the best overall forecasting accuracy. Although XGBoost achieved a slightly lower MAPE (12.51%), Prophet demonstrated more consistent performance across the major evaluation metrics and is therefore recommended as the final forecasting model.